# Notebook 03 — Data Cleaning & Standardization
## Section 1: Timestamp Standardization (Multi-Month)

**Why this section**  
Railway times must share one timezone and an hour key so they can join Open-Meteo weather.

**What we do (per month)**  
1. Load `ice_raw_YYYY-MM.parquet`  
2. Convert 5 timestamp columns → `Europe/Berlin`  
3. Create merge keys:  
   - `departure_planned_hour` (tz-aware, floored)  
   - `departure_planned_hour_naive` (for join with weather)  
4. Save `ice_standardized_YYYY-MM.parquet` + report  

**No rows dropped here** — only timezone / merge-key work.

**Outputs**  
- `ice_standardized_YYYY-MM.parquet`  
- `timestamp_standardize_report_YYYY-MM.json`  
- `timestamp_standardize_summary_multi_month.json`

In [2]:
# =============================================================================
# Notebook 03 | Section 1: Timestamp Standardization (Multi-Month)
# =============================================================================
# Self-contained: loads ice_raw per month; Europe/Berlin; merge hour keys;
# saves ice_standardized_YYYY-MM.parquet + reports.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"
MERGE_STRATEGY_PATH = REFERENCE_DIR / "merge_strategy.json"

TARGET_TZ = "Europe/Berlin"
TIMESTAMP_COLS = [
    "time",
    "arrival_planned_time",
    "arrival_change_time",
    "departure_planned_time",
    "departure_change_time",
]


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


config = load_json(CONFIG_PATH)
_ = load_json(MERGE_STRATEGY_PATH)  # must exist from Notebook 01
TARGET_MONTHS = config["scope"]["target_months"]
PRIMARY_TARGET = config["ml_tasks"]["primary"]["target"]


def standardize_to_berlin(series: pd.Series, tz: str = TARGET_TZ) -> tuple[pd.Series, int]:
    """Parse → localize/convert to Europe/Berlin. Returns series + new NaTs from DST."""
    parsed = pd.to_datetime(series, errors="coerce")
    nat_before = int(parsed.isna().sum())

    if parsed.dt.tz is None:
        localized = parsed.dt.tz_localize(
            tz,
            ambiguous="infer",
            nonexistent="shift_forward",
        )
    else:
        localized = parsed.dt.tz_convert(tz)

    nat_after = int(localized.isna().sum())
    return localized, nat_after - nat_before


def process_month(target_month: str) -> dict:
    input_parquet = PROCESSED_DIR / f"ice_raw_{target_month}.parquet"
    output_parquet = PROCESSED_DIR / f"ice_standardized_{target_month}.parquet"
    report_path = REFERENCE_DIR / f"timestamp_standardize_report_{target_month}.json"

    if not input_parquet.exists():
        raise FileNotFoundError(
            f"Missing: {input_parquet}\nRun Notebook 02 Section 1 first."
        )

    print("=" * 72)
    print(f"STANDARDIZE TIMESTAMPS — {target_month}")
    print("=" * 72)
    print(f"Loading: {input_parquet}")
    ice_df = pd.read_parquet(input_parquet)
    rows_before = len(ice_df)
    print(f"Rows: {rows_before:,}")
    print()

    dtypes_before = {col: str(ice_df[col].dtype) for col in TIMESTAMP_COLS}
    dst_issues = {}

    print(f"Converting to {TARGET_TZ}:")
    for col in TIMESTAMP_COLS:
        ice_df[col], new_nats = standardize_to_berlin(ice_df[col])
        if new_nats > 0:
            dst_issues[col] = int(new_nats)
        print(
            f"  {col:<28} → {ice_df[col].dtype}  "
            f"(null: {ice_df[col].isna().sum():,})"
        )
    print()

    ice_df["departure_planned_hour"] = ice_df["departure_planned_time"].dt.floor("h")
    ice_df["departure_planned_hour_naive"] = ice_df["departure_planned_hour"].dt.tz_localize(
        None
    )

    merge_key_nulls = int(ice_df["departure_planned_hour"].isna().sum())
    merge_key_null_pct = round(100 * merge_key_nulls / len(ice_df), 2)
    unique_hours = int(ice_df["departure_planned_hour_naive"].nunique(dropna=True))

    print("Merge keys created:")
    print(f"  departure_planned_hour       (tz-aware)")
    print(f"  departure_planned_hour_naive (for weather join)")
    print(f"  Null merge keys : {merge_key_nulls:,} ({merge_key_null_pct}%)")
    print(f"  Unique hours    : {unique_hours:,}")
    print()

    validations = {
        "row_count_unchanged": {
            "pass": len(ice_df) == rows_before,
            "before": rows_before,
            "after": len(ice_df),
        }
    }
    for col in TIMESTAMP_COLS:
        is_berlin = str(ice_df[col].dtype) == "datetime64[ns, Europe/Berlin]"
        validations[f"tz_aware_{col}"] = {
            "pass": bool(is_berlin),
            "dtype": str(ice_df[col].dtype),
        }

    failed = [
        k for k, v in validations.items() if isinstance(v, dict) and v.get("pass") is False
    ]
    if failed:
        raise ValueError(f"Validation failed for {target_month}: {failed}")

    sample_check = ice_df[ice_df["departure_planned_time"].notna()].head(3)
    rounding_examples = [
        {
            "departure_planned_time": str(row["departure_planned_time"]),
            "departure_planned_hour": str(row["departure_planned_hour"]),
            "departure_planned_hour_naive": str(row["departure_planned_hour_naive"]),
        }
        for _, row in sample_check.iterrows()
    ]

    ice_df.to_parquet(output_parquet, index=False)

    report = {
        "metadata": {
            "notebook": "03_Data_Cleaning_and_Standardization",
            "section": "Section 1",
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "target_month": target_month,
            "input_parquet": str(input_parquet),
            "output_parquet": str(output_parquet),
            "timezone": TARGET_TZ,
            "primary_target": PRIMARY_TARGET,
            "primary_metric": "mae",
        },
        "rows": {
            "before": rows_before,
            "after": int(len(ice_df)),
            "unchanged": True,
        },
        "dtypes_before": dtypes_before,
        "dtypes_after": {col: str(ice_df[col].dtype) for col in TIMESTAMP_COLS},
        "timestamp_null_counts": {
            col: int(ice_df[col].isna().sum()) for col in TIMESTAMP_COLS
        },
        "dst_issues_new_nats": dst_issues,
        "merge_key": {
            "column": "departure_planned_hour",
            "naive_column": "departure_planned_hour_naive",
            "null_count": merge_key_nulls,
            "null_pct": merge_key_null_pct,
            "unique_hours": unique_hours,
            "rounding_examples": rounding_examples,
        },
        "validations": validations,
        "validation_passed": len(failed) == 0,
    }

    with report_path.open("w", encoding="utf-8") as f:
        json.dump(make_json_safe(report), f, indent=2, ensure_ascii=False)

    print(f"Saved: {output_parquet}")
    print(f"Report: {report_path}")
    if rounding_examples:
        print("Example (minute → hour):")
        ex = rounding_examples[0]
        print(f"  {ex['departure_planned_time']}")
        print(f"  → {ex['departure_planned_hour_naive']}")
    print()

    return {
        "month": target_month,
        "rows": rows_before,
        "merge_key_null_pct": merge_key_null_pct,
        "unique_hours": unique_hours,
        "output_parquet": str(output_parquet),
        "report": str(report_path),
    }


# =============================================================================
# RUN ALL MONTHS
# =============================================================================
print("Notebook 03 | Section 1 — Multi-month timestamp standardization")
print(f"Months : {', '.join(TARGET_MONTHS)}")
print(f"TZ     : {TARGET_TZ}")
print(f"Target : {PRIMARY_TARGET} | Metric: MAE")
print()

month_results = [process_month(m) for m in TARGET_MONTHS]

summary_path = REFERENCE_DIR / "timestamp_standardize_summary_multi_month.json"
summary = {
    "metadata": {
        "notebook": "03_Data_Cleaning_and_Standardization",
        "section": "Section 1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_months": TARGET_MONTHS,
        "timezone": TARGET_TZ,
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
    },
    "months": month_results,
    "totals": {
        "rows_all_months": int(sum(r["rows"] for r in month_results)),
    },
}

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(summary), f, indent=2, ensure_ascii=False)

sep = "=" * 72
print(sep)
print("Section 1 COMPLETE")
print(sep)
print(f"{'Month':<10} {'Rows':>12} {'Merge-key null%':>16} {'Unique hours':>14}")
for r in month_results:
    print(
        f"{r['month']:<10} {r['rows']:>12,} "
        f"{r['merge_key_null_pct']:>15.2f}% {r['unique_hours']:>14,}"
    )
print()
print(f"Summary: {summary_path}")
print(sep)
print("Next: Section 2 — drop cancellations, handle nulls, save ice_cleaned")
print(sep)

Notebook 03 | Section 1 — Multi-month timestamp standardization
Months : 2024-07, 2024-08, 2024-09
TZ     : Europe/Berlin
Target : delay_in_min | Metric: MAE

STANDARDIZE TIMESTAMPS — 2024-07
Loading: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\ice_raw_2024-07.parquet
Rows: 157,886

Converting to Europe/Berlin:
  time                         → datetime64[ns, Europe/Berlin]  (null: 0)
  arrival_planned_time         → datetime64[ns, Europe/Berlin]  (null: 15,310)
  arrival_change_time          → datetime64[ns, Europe/Berlin]  (null: 15,307)
  departure_planned_time       → datetime64[ns, Europe/Berlin]  (null: 15,452)
  departure_change_time        → datetime64[ns, Europe/Berlin]  (null: 15,441)

Merge keys created:
  departure_planned_hour       (tz-aware)
  departure_planned_hour_naive (for weather join)
  Null merge keys : 15,452 (9.79%)
  Unique hours    : 744

Saved: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\ice_standardized_2024-07.parquet
Re

# Notebook 03 — Data Cleaning & Standardization
## Section 2: Cleaning Rules & Cleaned Export (Multi-Month)

**Why this section**  
Canceled stops have no real delay outcome. We clean before weather merge and modeling.

**Rules (per month)**  

| Rule | Action | Reason |
|------|--------|--------|
| `is_canceled == True` | **Drop** | No valid `delay_in_min` |
| Duplicate `id` | **Drop** | Avoid double-counting |
| Null `station_name` | Fill from `xml_station_name` | Label only |
| Null `line_number` | **Keep** | Normal for ICE |
| Null departure time | Keep row; set `mergeable=False` | Weather join later |

**Target stays continuous:** `delay_in_min` (regression). Primary metric later: **MAE**.

**Outputs**  
- `ice_cleaned_YYYY-MM.parquet`  
- `cleaning_report_YYYY-MM.json`  
- `cleaning_summary_multi_month.json`

In [4]:
# =============================================================================
# Notebook 03 | Section 2: Cleaning (Multi-Month)
# =============================================================================
# Load ice_standardized → drop canceled/duplicates → fill station names →
# flag mergeable → save ice_cleaned + reports. Regression target = delay_in_min.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"

TIMESTAMP_COLS = [
    "time",
    "arrival_planned_time",
    "arrival_change_time",
    "departure_planned_time",
    "departure_change_time",
]


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if pd.isna(obj):
        return None
    return obj


config = load_json(CONFIG_PATH)
TARGET_MONTHS = config["scope"]["target_months"]
PRIMARY_TARGET = config["ml_tasks"]["primary"]["target"]


def process_month(target_month: str) -> dict:
    input_parquet = PROCESSED_DIR / f"ice_standardized_{target_month}.parquet"
    output_parquet = PROCESSED_DIR / f"ice_cleaned_{target_month}.parquet"
    report_path = REFERENCE_DIR / f"cleaning_report_{target_month}.json"

    if not input_parquet.exists():
        raise FileNotFoundError(
            f"Missing: {input_parquet}\nRun Notebook 03 Section 1 first."
        )

    print("=" * 72)
    print(f"CLEAN ICE DATA — {target_month}")
    print("=" * 72)

    ice_df = pd.read_parquet(input_parquet)
    rows_in = len(ice_df)
    cleaning_log = []
    print(f"Loaded: {rows_in:,} rows from {input_parquet.name}")
    print()

    # --- 1) ICE only (should already be ICE from Notebook 02) ---
    before = len(ice_df)
    ice_df = ice_df[ice_df["train_type"] == "ICE"].copy()
    cleaning_log.append(
        {
            "step": "filter_ICE_only",
            "rows_before": before,
            "rows_after": len(ice_df),
            "rows_removed": before - len(ice_df),
            "detail": "Keep train_type == ICE only",
        }
    )

    # --- 2) Drop canceled stops (no valid delay outcome) ---
    before = len(ice_df)
    n_canceled = int(ice_df["is_canceled"].sum())
    ice_df = ice_df[~ice_df["is_canceled"]].copy()
    cleaning_log.append(
        {
            "step": "exclude_canceled",
            "rows_before": before,
            "rows_after": len(ice_df),
            "rows_removed": before - len(ice_df),
            "detail": f"{n_canceled:,} canceled stops removed",
        }
    )
    print(f"Dropped canceled : {n_canceled:,}  →  {len(ice_df):,} rows left")

    # --- 3) Drop duplicate ids ---
    before = len(ice_df)
    n_dup = int(ice_df["id"].duplicated().sum())
    ice_df = ice_df.drop_duplicates(subset=["id"], keep="first")
    cleaning_log.append(
        {
            "step": "drop_duplicate_ids",
            "rows_before": before,
            "rows_after": len(ice_df),
            "rows_removed": before - len(ice_df),
            "detail": f"{n_dup:,} duplicate ids removed",
        }
    )
    print(f"Dropped duplicates: {n_dup:,}")

    # --- 4) Fill station_name from xml_station_name ---
    null_before = int(ice_df["station_name"].isna().sum())
    mask = ice_df["station_name"].isna() & ice_df["xml_station_name"].notna()
    ice_df.loc[mask, "station_name"] = ice_df.loc[mask, "xml_station_name"]
    null_after = int(ice_df["station_name"].isna().sum())
    cleaning_log.append(
        {
            "step": "fill_station_name",
            "nulls_before": null_before,
            "nulls_after": null_after,
            "detail": "Filled from xml_station_name where station_name was null",
        }
    )
    print(f"station_name nulls: {null_before:,} → {null_after:,}")

    # --- 5) line_number nulls kept (expected for ICE) ---
    line_null_pct = round(100 * ice_df["line_number"].isna().mean(), 2)
    cleaning_log.append(
        {
            "step": "line_number_nulls_kept",
            "null_pct": line_null_pct,
            "detail": "ICE rows often have null line_number — expected, rows kept",
        }
    )
    print(f"line_number null %: {line_null_pct}% (kept)")

    # --- 6) Mergeable flag (need departure hour for weather join) ---
    ice_df["mergeable"] = ice_df["departure_planned_hour_naive"].notna()
    mergeable_rows = int(ice_df["mergeable"].sum())
    non_mergeable = int((~ice_df["mergeable"]).sum())
    non_mergeable_pct = round(100 * non_mergeable / len(ice_df), 2)
    print(
        f"Mergeable rows   : {mergeable_rows:,}  "
        f"(non-mergeable: {non_mergeable:,} = {non_mergeable_pct}%)"
    )
    print()

    # --- Delay target stats (regression, continuous minutes) ---
    delay = ice_df[PRIMARY_TARGET]
    delay_stats = {
        "count": int(delay.notna().sum()),
        "mean": round(float(delay.mean()), 2),
        "median": round(float(delay.median()), 2),
        "std": round(float(delay.std()), 2),
        "min": int(delay.min()),
        "max": int(delay.max()),
        "p25": round(float(delay.quantile(0.25)), 2),
        "p75": round(float(delay.quantile(0.75)), 2),
    }
    print(f"Target `{PRIMARY_TARGET}` after cleaning:")
    print(
        f"  mean={delay_stats['mean']} min | "
        f"median={delay_stats['median']} min | "
        f"range=[{delay_stats['min']}, {delay_stats['max']}]"
    )
    print()

    # --- Validations ---
    validations = {
        "only_ICE": {
            "pass": bool((ice_df["train_type"] == "ICE").all()),
            "non_ICE_count": int((ice_df["train_type"] != "ICE").sum()),
        },
        "no_canceled": {
            "pass": bool(ice_df["is_canceled"].sum() == 0),
            "canceled_count": int(ice_df["is_canceled"].sum()),
        },
        "no_duplicate_ids": {
            "pass": bool(not ice_df["id"].duplicated().any()),
            "duplicate_count": int(ice_df["id"].duplicated().sum()),
        },
        "station_name_nulls": {
            "pass": bool(ice_df["station_name"].isna().sum() == 0),
            "null_count": int(ice_df["station_name"].isna().sum()),
        },
        "timestamps_tz_aware": {
            "pass": all(
                str(ice_df[c].dtype) == "datetime64[ns, Europe/Berlin]"
                for c in TIMESTAMP_COLS
                if c in ice_df.columns
            ),
            "dtype": str(ice_df["departure_planned_time"].dtype),
        },
        "merge_key_present": {
            "pass": "departure_planned_hour_naive" in ice_df.columns,
        },
        "target_present": {
            "pass": PRIMARY_TARGET in ice_df.columns
            and int(ice_df[PRIMARY_TARGET].isna().sum()) == 0,
            "null_count": int(ice_df[PRIMARY_TARGET].isna().sum())
            if PRIMARY_TARGET in ice_df.columns
            else None,
        },
    }

    failed = [
        k for k, v in validations.items() if isinstance(v, dict) and v.get("pass") is False
    ]
    if failed:
        raise ValueError(f"Validation failed for {target_month}: {failed}")

    rows_out = len(ice_df)
    removed = rows_in - rows_out
    removal_pct = round(100 * removed / rows_in, 2)

    ice_df.to_parquet(output_parquet, index=False)

    report = {
        "metadata": {
            "notebook": "03_Data_Cleaning_and_Standardization",
            "section": "Section 2",
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "target_month": target_month,
            "input_parquet": str(input_parquet),
            "output_parquet": str(output_parquet),
            "primary_target": PRIMARY_TARGET,
            "primary_metric": "mae",
            "task_type": "regression",
        },
        "rows": {
            "input": rows_in,
            "output": rows_out,
            "removed": removed,
            "removal_pct": removal_pct,
        },
        "cleaning_log": cleaning_log,
        "mergeable": {
            "mergeable_rows": mergeable_rows,
            "non_mergeable_rows": non_mergeable,
            "non_mergeable_pct": non_mergeable_pct,
            "note": (
                "Non-mergeable rows have no departure_planned_time — "
                "excluded from weather join in Notebook 04"
            ),
        },
        "line_number_null_pct": line_null_pct,
        "delay_stats_cleaned": delay_stats,
        "validations": validations,
        "validation_passed": len(failed) == 0,
        "columns_in_output": list(ice_df.columns),
        "ready_for_notebook_04": True,
    }

    with report_path.open("w", encoding="utf-8") as f:
        json.dump(make_json_safe(report), f, indent=2, ensure_ascii=False)

    print(f"Saved : {output_parquet}")
    print(f"Report: {report_path}")
    print(f"Rows  : {rows_in:,} → {rows_out:,}  (removed {removed:,} = {removal_pct}%)")
    print()

    return {
        "month": target_month,
        "rows_in": rows_in,
        "rows_out": rows_out,
        "removed": removed,
        "removal_pct": removal_pct,
        "canceled_removed": n_canceled,
        "mergeable_rows": mergeable_rows,
        "non_mergeable_pct": non_mergeable_pct,
        "delay_mean": delay_stats["mean"],
        "delay_median": delay_stats["median"],
        "output_parquet": str(output_parquet),
        "report": str(report_path),
    }


# =============================================================================
# RUN ALL MONTHS
# =============================================================================
print("Notebook 03 | Section 2 — Multi-month cleaning")
print(f"Months : {', '.join(TARGET_MONTHS)}")
print(f"Target : {PRIMARY_TARGET} (regression) | Metric later: MAE")
print()

month_results = [process_month(m) for m in TARGET_MONTHS]

summary_path = REFERENCE_DIR / "cleaning_summary_multi_month.json"
summary = {
    "metadata": {
        "notebook": "03_Data_Cleaning_and_Standardization",
        "section": "Section 2",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_months": TARGET_MONTHS,
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
        "task_type": "regression",
    },
    "months": month_results,
    "totals": {
        "rows_in_all": int(sum(r["rows_in"] for r in month_results)),
        "rows_out_all": int(sum(r["rows_out"] for r in month_results)),
        "removed_all": int(sum(r["removed"] for r in month_results)),
        "mergeable_all": int(sum(r["mergeable_rows"] for r in month_results)),
    },
}

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(summary), f, indent=2, ensure_ascii=False)

sep = "=" * 72
print(sep)
print("Section 2 COMPLETE")
print(sep)
print(
    f"{'Month':<10} {'In':>10} {'Out':>10} {'Removed':>10} "
    f"{'Mean delay':>12} {'Median':>8}"
)
for r in month_results:
    print(
        f"{r['month']:<10} {r['rows_in']:>10,} {r['rows_out']:>10,} "
        f"{r['removed']:>10,} {r['delay_mean']:>11.1f}m {r['delay_median']:>7.1f}m"
    )
print()
print(
    f"TOTAL cleaned rows : {summary['totals']['rows_out_all']:,}  "
    f"(from {summary['totals']['rows_in_all']:,})"
)
print(f"Summary file       : {summary_path}")
print(sep)
print("Next: Section 3 — Notebook 03 close-out checklist")
print(sep)

Notebook 03 | Section 2 — Multi-month cleaning
Months : 2024-07, 2024-08, 2024-09
Target : delay_in_min (regression) | Metric later: MAE

CLEAN ICE DATA — 2024-07
Loaded: 157,886 rows from ice_standardized_2024-07.parquet

Dropped canceled : 11,497  →  146,389 rows left
Dropped duplicates: 0
station_name nulls: 0 → 0
line_number null %: 100.0% (kept)
Mergeable rows   : 132,268  (non-mergeable: 14,121 = 9.65%)

Target `delay_in_min` after cleaning:
  mean=10.92 min | median=4.0 min | range=[-46, 298]

Saved : c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\ice_cleaned_2024-07.parquet
Report: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\cleaning_report_2024-07.json
Rows  : 157,886 → 146,389  (removed 11,497 = 7.28%)

CLEAN ICE DATA — 2024-08
Loaded: 150,289 rows from ice_standardized_2024-08.parquet

Dropped canceled : 13,968  →  136,321 rows left
Dropped duplicates: 0
station_name nulls: 0 → 0
line_number null %: 100.0% (kept)
Mergeable rows   : 122,812 

# Notebook 03 — Data Cleaning & Standardization
## Section 3: Cleaning Summary & Close-Out

**Why this section**  
Confirm every Notebook 03 file is on disk and that cleaned ICE data is ready for weather merge.

**What we check**  
1. Standardized + cleaned Parquets for Jul / Aug / Sep  
2. Timestamp + cleaning reports  
3. Row counts before → after  

**Next**  
Notebook 04: geocode stations → fetch weather → **save merged Parquet on disk**.

In [6]:
# =============================================================================
# Notebook 03 | Section 3: Close-Out (Multi-Month)
# =============================================================================
# Verify standardized + cleaned files exist; summarize; save notebook_03_summary.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


config = load_json(CONFIG_PATH)
TARGET_MONTHS = config["scope"]["target_months"]
PRIMARY_TARGET = config["ml_tasks"]["primary"]["target"]

print("Notebook 03 | Section 3 — Close-out checklist")
print(f"Months : {', '.join(TARGET_MONTHS)}")
print(f"Target : {PRIMARY_TARGET} (regression) | Metric later: MAE")
print()

# ---------------------------------------------------------------------------
# FILE CHECKLIST
# ---------------------------------------------------------------------------
checklist = []

def check(label: str, path: Path) -> bool:
    ok = path.exists()
    size = path.stat().st_size if ok else 0
    checklist.append(
        {
            "label": label,
            "path": str(path),
            "exists": ok,
            "size_bytes": int(size),
            "size_mb": round(size / 1e6, 2) if ok else 0.0,
        }
    )
    status = "OK" if ok else "MISSING"
    size_txt = f"{size / 1e6:.1f} MB" if ok else "—"
    print(f"  [{status}] {label:<42} {size_txt}")
    return ok


print("=" * 72)
print("FILE CHECKLIST")
print("=" * 72)

all_ok = True
for month in TARGET_MONTHS:
    print(f"\n{month}")
    all_ok &= check(
        f"ice_standardized_{month}.parquet",
        PROCESSED_DIR / f"ice_standardized_{month}.parquet",
    )
    all_ok &= check(
        f"ice_cleaned_{month}.parquet",
        PROCESSED_DIR / f"ice_cleaned_{month}.parquet",
    )
    all_ok &= check(
        f"timestamp_standardize_report_{month}.json",
        REFERENCE_DIR / f"timestamp_standardize_report_{month}.json",
    )
    all_ok &= check(
        f"cleaning_report_{month}.json",
        REFERENCE_DIR / f"cleaning_report_{month}.json",
    )

print()
all_ok &= check(
    "timestamp_standardize_summary_multi_month.json",
    REFERENCE_DIR / "timestamp_standardize_summary_multi_month.json",
)
all_ok &= check(
    "cleaning_summary_multi_month.json",
    REFERENCE_DIR / "cleaning_summary_multi_month.json",
)

# ---------------------------------------------------------------------------
# LOAD SUMMARIES + QUICK ROW COUNTS FROM PARQUET
# ---------------------------------------------------------------------------
ts_summary = load_json(REFERENCE_DIR / "timestamp_standardize_summary_multi_month.json")
clean_summary = load_json(REFERENCE_DIR / "cleaning_summary_multi_month.json")

print()
print("=" * 72)
print("ROW COUNTS (from reports + disk)")
print("=" * 72)
print(
    f"{'Month':<10} {'Standardized':>14} {'Cleaned':>12} {'Removed':>10} "
    f"{'Mean delay':>12}"
)

month_detail = []
for month in TARGET_MONTHS:
    cleaned_path = PROCESSED_DIR / f"ice_cleaned_{month}.parquet"
    n_disk = int(len(pd.read_parquet(cleaned_path, columns=["id"]))) if cleaned_path.exists() else None

    c = next(m for m in clean_summary["months"] if m["month"] == month)
    print(
        f"{month:<10} {c['rows_in']:>14,} {c['rows_out']:>12,} "
        f"{c['removed']:>10,} {c['delay_mean']:>11.1f}m"
    )
    if n_disk is not None and n_disk != c["rows_out"]:
        raise ValueError(
            f"Row mismatch {month}: report={c['rows_out']:,}, disk={n_disk:,}"
        )
    month_detail.append(
        {
            "month": month,
            "standardized_rows": c["rows_in"],
            "cleaned_rows": c["rows_out"],
            "removed": c["removed"],
            "removal_pct": c["removal_pct"],
            "mergeable_rows": c["mergeable_rows"],
            "delay_mean": c["delay_mean"],
            "delay_median": c["delay_median"],
            "cleaned_parquet": str(cleaned_path),
            "cleaned_rows_on_disk": n_disk,
        }
    )

totals = clean_summary["totals"]
print()
print(f"TOTAL standardized : {totals['rows_in_all']:,}")
print(f"TOTAL cleaned      : {totals['rows_out_all']:,}")
print(f"TOTAL removed      : {totals['removed_all']:,}  (mostly canceled)")
print(f"TOTAL mergeable    : {totals['mergeable_all']:,}  (have departure hour)")
print()

# ---------------------------------------------------------------------------
# PIPELINE RECAP
# ---------------------------------------------------------------------------
print("=" * 72)
print("WHAT NOTEBOOK 03 DID")
print("=" * 72)
print("  1. Timestamps → Europe/Berlin + hour merge keys")
print("  2. Dropped canceled stops (no valid delay)")
print("  3. Dropped duplicate ids; filled station_name")
print("  4. Saved ice_cleaned_YYYY-MM.parquet for each month")
print("  5. Target stays continuous: delay_in_min (regression → MAE later)")
print()

ready = bool(all_ok)
print("=" * 72)
print(f"Ready for Notebook 04 (geocode + weather + MERGE): {'YES' if ready else 'NO'}")
print("=" * 72)
if not ready:
    missing = [c["label"] for c in checklist if not c["exists"]]
    raise FileNotFoundError(f"Missing files: {missing}")

# ---------------------------------------------------------------------------
# SAVE CLOSE-OUT SUMMARY
# ---------------------------------------------------------------------------
summary_path = REFERENCE_DIR / "notebook_03_summary.json"
summary = {
    "metadata": {
        "notebook": "03_Data_Cleaning_and_Standardization",
        "section": "Section 3",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_months": TARGET_MONTHS,
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
        "task_type": "regression",
    },
    "checklist": checklist,
    "all_files_ok": ready,
    "months": month_detail,
    "totals": totals,
    "timestamp_summary_file": str(
        REFERENCE_DIR / "timestamp_standardize_summary_multi_month.json"
    ),
    "cleaning_summary_file": str(REFERENCE_DIR / "cleaning_summary_multi_month.json"),
    "cleaned_data_location": str(PROCESSED_DIR),
    "cleaned_files": [
        f"ice_cleaned_{m}.parquet" for m in TARGET_MONTHS
    ],
    "next_notebook": {
        "id": "04",
        "goal": "Geocode eva → fetch Open-Meteo weather → left-join → save merged Parquet on disk",
        "inputs": [f"ice_cleaned_{m}.parquet" for m in TARGET_MONTHS],
        "expected_outputs": [
            "station_coordinates.json",
            "weather_by_station_YYYY-MM.parquet",
            "ice_weather_merged_YYYY-MM.parquet",
        ],
    },
    "ready_for_notebook_04": ready,
}

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(summary), f, indent=2, ensure_ascii=False)

print()
print(f"Saved close-out: {summary_path}")
print()
print("Cleaned data folder:")
print(f"  {PROCESSED_DIR}")
for m in TARGET_MONTHS:
    print(f"  → ice_cleaned_{m}.parquet")
print()
print("Next: Notebook 04 — geocode, weather fetch, merge, save merged Parquet")

Notebook 03 | Section 3 — Close-out checklist
Months : 2024-07, 2024-08, 2024-09
Target : delay_in_min (regression) | Metric later: MAE

FILE CHECKLIST

2024-07
  [OK] ice_standardized_2024-07.parquet           5.7 MB
  [OK] ice_cleaned_2024-07.parquet                5.5 MB
  [OK] timestamp_standardize_report_2024-07.json  0.0 MB
  [OK] cleaning_report_2024-07.json               0.0 MB

2024-08
  [OK] ice_standardized_2024-08.parquet           5.5 MB
  [OK] ice_cleaned_2024-08.parquet                5.1 MB
  [OK] timestamp_standardize_report_2024-08.json  0.0 MB
  [OK] cleaning_report_2024-08.json               0.0 MB

2024-09
  [OK] ice_standardized_2024-09.parquet           5.4 MB
  [OK] ice_cleaned_2024-09.parquet                5.1 MB
  [OK] timestamp_standardize_report_2024-09.json  0.0 MB
  [OK] cleaning_report_2024-09.json               0.0 MB

  [OK] timestamp_standardize_summary_multi_month.json 0.0 MB
  [OK] cleaning_summary_multi_month.json          0.0 MB

ROW COUNTS (from 